# Experiment 02 — Spatial Leakage Localization
**Date:** 2026-09-14

This notebook analyzes the stored HUGS Baseline v1 NPZ outputs without rerunning HUGS.

Primary question:

> Where are the non-target Gaussians responsible for the high distal-joint leakage, and which SMPL joint assignments dominate those displaced Gaussians?

Primary comparison:
- left ankle z, +10°
- left wrist z, +10°
- left shoulder z, +10°

The raw NPZ files remain in Google Drive. GPU is not required for this analysis.


## Step 1 — Mount Drive and verify the three NPZ files

Run this cell first. Stop here and inspect the output before continuing.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import numpy as np

ROOT = Path(
    "/content/drive/MyDrive/interactive-digital-humans/"
    "experiments/01-baseline/probe_results"
)

TARGETS = {
    "ankle_z": ROOT / "frame000_left_ankle_z_10deg.npz",
    "wrist_z": ROOT / "frame000_left_wrist_z_10deg.npz",
    "shoulder_z": ROOT / "frame000_left_shoulder_z_10deg.npz",
}

print("Probe folder exists:", ROOT.exists())
print("Probe folder:", ROOT)

for label, path in TARGETS.items():
    print("\n" + "=" * 72)
    print(label)
    print("file:", path.name)
    print("exists:", path.exists())
    if path.exists():
        with np.load(path) as d:
            print("arrays:", d.files)
            for key in d.files:
                arr = d[key]
                print(f"{key:18s} shape={arr.shape} dtype={arr.dtype}")


## Step 2 — Load the probes and validate comparable Gaussian indexing

Only run this after Step 1 confirms all three files exist.


In [ ]:
PROBES = {}

for label, path in TARGETS.items():
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path) as d:
        PROBES[label] = {k: d[k].copy() for k in d.files}

required = {
    "xyz_before", "xyz_after", "xyz_canon",
    "dominant_joint", "joint_confidence", "nearest_vertex"
}

for label, d in PROBES.items():
    missing = required - set(d)
    assert not missing, f"{label}: missing arrays {missing}"
    n = d["xyz_before"].shape[0]
    assert d["xyz_after"].shape == (n, 3)
    assert d["xyz_canon"].shape == (n, 3)
    assert d["dominant_joint"].shape[0] == n
    assert d["joint_confidence"].shape[0] == n
    print(label, "N =", n)

sizes = {d["xyz_before"].shape[0] for d in PROBES.values()}
assert len(sizes) == 1, f"Probe sizes differ: {sizes}"

ref = PROBES["ankle_z"]["xyz_canon"]
for label, d in PROBES.items():
    max_err = float(np.max(np.abs(d["xyz_canon"] - ref)))
    print(label, "max canonical-coordinate difference:", max_err)


## Step 3 — Define SMPL target subtrees and recompute leakage

Expected descendants are included in the target subtree so normal child-joint motion is not mislabeled as leakage.


In [ ]:
SMPL_JOINT_NAMES = [
    "pelvis",
    "left_hip", "right_hip", "spine1",
    "left_knee", "right_knee", "spine2",
    "left_ankle", "right_ankle", "spine3",
    "left_foot", "right_foot", "neck",
    "left_collar", "right_collar", "head",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hand", "right_hand",
]

JOINT_ID = {name: i for i, name in enumerate(SMPL_JOINT_NAMES)}

TARGET_SUBTREES = {
    "ankle_z": {JOINT_ID["left_ankle"], JOINT_ID["left_foot"]},
    "wrist_z": {JOINT_ID["left_wrist"], JOINT_ID["left_hand"]},
    "shoulder_z": {
        JOINT_ID["left_shoulder"],
        JOINT_ID["left_elbow"],
        JOINT_ID["left_wrist"],
        JOINT_ID["left_hand"],
    },
}

def displacement(d):
    return np.linalg.norm(d["xyz_after"] - d["xyz_before"], axis=1)

def summarize_probe(label, d, confidence_threshold=0.0):
    disp = displacement(d)
    joint = d["dominant_joint"].astype(int)
    conf = d["joint_confidence"]

    valid = conf >= confidence_threshold
    target_mask = np.isin(joint, list(TARGET_SUBTREES[label]))
    target = target_mask & valid
    outside = (~target_mask) & valid

    target_change = float(disp[target].sum())
    outside_change = float(disp[outside].sum())
    denom = target_change + outside_change
    leakage = outside_change / denom if denom > 0 else float("nan")

    return {
        "probe": label,
        "confidence_threshold": confidence_threshold,
        "num_valid": int(valid.sum()),
        "target_change": target_change,
        "outside_change": outside_change,
        "leakage_ratio": leakage,
        "leakage_pct": leakage * 100.0,
        "outside_p99": float(np.quantile(disp[outside], 0.99)) if outside.any() else float("nan"),
    }

for threshold in (0.0, 0.9):
    print("\nconfidence >=", threshold)
    for label, d in PROBES.items():
        row = summarize_probe(label, d, threshold)
        print(
            label,
            f"leakage={row['leakage_pct']:.4f}%",
            f"outside_p99={row['outside_p99']:.6f}",
            f"valid={row['num_valid']}"
        )


## Step 4 — Attribute outside displacement to SMPL joints

This turns the global leakage percentage into a joint-by-joint breakdown.


In [ ]:
import pandas as pd

rows = []

for label, d in PROBES.items():
    disp = displacement(d)
    joint = d["dominant_joint"].astype(int)
    conf = d["joint_confidence"]
    outside = ~np.isin(joint, list(TARGET_SUBTREES[label]))

    total_outside = disp[outside].sum()

    for jid in np.unique(joint[outside]):
        mask = outside & (joint == jid)
        s = float(disp[mask].sum())
        rows.append({
            "probe": label,
            "joint_id": int(jid),
            "joint": SMPL_JOINT_NAMES[int(jid)] if int(jid) < len(SMPL_JOINT_NAMES) else f"joint_{jid}",
            "num_gaussians": int(mask.sum()),
            "displacement_sum": s,
            "fraction_of_outside_pct": 100.0 * s / total_outside if total_outside > 0 else 0.0,
            "mean_confidence": float(conf[mask].mean()) if mask.any() else float("nan"),
        })

joint_breakdown = pd.DataFrame(rows)

for label in TARGETS:
    print("\n", label)
    display(
        joint_breakdown[joint_breakdown["probe"] == label]
        .sort_values("displacement_sum", ascending=False)
        .head(12)
        .reset_index(drop=True)
    )


## Step 5 — Plot the highest-displacement non-target Gaussians

This is the first spatial localization view. It helps determine whether leakage clusters near anatomical boundaries or is spatially distributed.


In [ ]:
import matplotlib.pyplot as plt

def plot_spatial_leakage(label, d, top_k=3000, background_sample=25000):
    xyz = d["xyz_before"]
    disp = displacement(d)
    joint = d["dominant_joint"].astype(int)
    outside = ~np.isin(joint, list(TARGET_SUBTREES[label]))

    outside_ids = np.flatnonzero(outside)
    order = outside_ids[np.argsort(disp[outside_ids])[-min(top_k, len(outside_ids)):]]

    rng = np.random.default_rng(20260914)
    if len(xyz) > background_sample:
        bg = rng.choice(len(xyz), background_sample, replace=False)
    else:
        bg = np.arange(len(xyz))

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(xyz[bg, 0], xyz[bg, 1], xyz[bg, 2], s=0.25, alpha=0.08)
    sc = ax.scatter(
        xyz[order, 0], xyz[order, 1], xyz[order, 2],
        c=disp[order], s=4, alpha=0.8
    )
    ax.set_title(f"{label}: highest-displacement non-target Gaussians")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    fig.colorbar(sc, ax=ax, shrink=0.65, label="displacement")
    plt.show()

for label, d in PROBES.items():
    plot_spatial_leakage(label, d)


## Step 6 — Save reviewed compact tables to Drive

Run only after the numbers and plots have been checked.


In [ ]:
OUT = Path(
    "/content/drive/MyDrive/interactive-digital-humans/"
    "experiments/02-spatial-leakage-localization"
)
OUT.mkdir(parents=True, exist_ok=True)

summary_rows = []
for threshold in (0.0, 0.9):
    for label, d in PROBES.items():
        summary_rows.append(summarize_probe(label, d, threshold))

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT / "spatial_leakage_summary.csv", index=False)
joint_breakdown.to_csv(OUT / "outside_joint_breakdown.csv", index=False)

print("Saved:", OUT / "spatial_leakage_summary.csv")
print("Saved:", OUT / "outside_joint_breakdown.csv")
print("Drive output folder:", OUT)
